# 04 Model Bias and Overfit Analysis

This notebook inspects train/validation/test behavior using saved report files and rolling CV outputs.

In [1]:
from pathlib import Path
import pandas as pd

REPORTS_DIR = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling/outputs/reports')

In [2]:
metrics = pd.concat([
    pd.read_csv(REPORTS_DIR / 'ds_workflow_a_metrics.csv'),
    pd.read_csv(REPORTS_DIR / 'ds_workflow_b_metrics.csv'),
    pd.read_csv(REPORTS_DIR / 'ds_workflow_c_metrics.csv'),
], ignore_index=True)
metrics.head()

,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
0,A,ETS,val,1,103,0.106033,4338.880526,1290.401337,0.858919,1500.984630,0.398058
1,A,ETS,val,2,103,0.091652,3796.748621,-81.702897,0.848642,1363.028451,0.485437
2,A,ETS,val,3,103,0.119636,5017.963034,-592.214897,1.010401,1813.327238,0.601942
3,A,ETS,val,4,103,0.104865,4871.055926,-1967.018160,0.964295,1664.670030,0.640777
4,A,ETS,val,5,103,0.157099,7265.007619,-3902.685500,1.452651,2650.603120,0.757282


## Overall Test Ranking

In [3]:
metrics[(metrics['split']=='test') & (metrics['horizon']==0)].sort_values(['dataset','WAPE','MASE_mean','RMSE'])

,dataset,model,split,horizon,n_obs,WAPE,RMSE,Bias,MASE_mean,wQL50,under_forecast_rate
39,A,ARIMA,test,0,1236,1.066327e-01,4.788504e+03,-3.008365e+02,9.522145e-01,1.646522e+03,0.521845
111,A,CATBOOST,test,0,14832,1.162810e-01,5.337523e+03,-1.238852e+03,1.038602e+00,1.795501e+03,0.548139
85,A,XGBOOST,test,0,14832,1.173748e-01,5.378519e+03,-1.040791e+03,1.032125e+00,1.812390e+03,0.573894
19,A,ETS,test,0,1236,1.182009e-01,5.550639e+03,-1.956369e+02,1.037084e+00,1.825147e+03,0.530744
59,A,SARIMA,test,0,1236,2.173242e-01,1.382771e+04,5.540652e+02,1.953624e+00,3.355715e+03,0.444984
151,B,ARIMA,test,0,1236,1.066327e-01,4.788504e+03,-3.008365e+02,9.522145e-01,1.646522e+03,0.521845
197,B,XGBOOST,test,0,14832,1.165232e-01,5.349682e+03,-1.065375e+03,1.025741e+00,1.799241e+03,0.569512
131,B,ETS,test,0,1236,1.182009e-01,5.550639e+03,-1.956369e+02,1.037084e+00,1.825147e+03,0.530744
223,B,CATBOOST,test,0,14832,1.196114e-01,5.499271e+03,-1.244579e+03,1.084185e+00,1.846926e+03,0.549083
171,B,SARIMA,test,0,1236,2.173242e-01,1.382771e+04,5.540652e+02,1.953624e+00,3.355715e+03,0.444984


## Bias Review

In [4]:
metrics[(metrics['split']=='test') & (metrics['horizon']==0)][['dataset','model','Bias','under_forecast_rate']].sort_values(['dataset','Bias'])

,dataset,model,Bias,under_forecast_rate
111,A,CATBOOST,-1.238852e+03,0.548139
85,A,XGBOOST,-1.040791e+03,0.573894
39,A,ARIMA,-3.008365e+02,0.521845
19,A,ETS,-1.956369e+02,0.530744
59,A,SARIMA,5.540652e+02,0.444984
223,B,CATBOOST,-1.244579e+03,0.549083
197,B,XGBOOST,-1.065375e+03,0.569512
151,B,ARIMA,-3.008365e+02,0.521845
131,B,ETS,-1.956369e+02,0.530744
171,B,SARIMA,5.540652e+02,0.444984


## Horizon Drift

In [5]:
metrics[(metrics['split']=='test') & (metrics['horizon'] > 0)][['dataset','model','horizon','WAPE','MASE_mean','RMSE']].sort_values(['dataset','model','horizon']).head(50)

,dataset,model,horizon,WAPE,MASE_mean,RMSE
27,A,ARIMA,1,0.103134,0.952410,4538.734715
28,A,ARIMA,2,0.085931,0.794905,3869.237750
29,A,ARIMA,3,0.105753,0.964783,4988.954718
30,A,ARIMA,4,0.109936,0.883611,5040.740047
31,A,ARIMA,5,0.109124,0.936320,4374.610610
32,A,ARIMA,6,0.099776,0.890714,3916.437095
33,A,ARIMA,7,0.113151,0.956304,5016.023145
34,A,ARIMA,8,0.108020,0.946498,4751.820209
35,A,ARIMA,9,0.105053,0.962114,4614.138591
36,A,ARIMA,10,0.098606,0.940400,4209.723644


## Rolling CV Summary

Run `rolling_cv.py` before this cell if the summary file does not exist.

In [6]:
cv_path = REPORTS_DIR / 'rolling_cv_metrics_summary.csv'
if cv_path.exists():
    pd.read_csv(cv_path).head(30)
else:
    print('Missing rolling_cv_metrics_summary.csv. Run rolling_cv.py first.')